In [ ]:
import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler, MinMaxScaler
import dbcv  

df = pd.read_csv("df.csv")  

def normal(col, skew_thr=0.5, kurt_thr=1.0):
    s = pd.Series(col).dropna()
    if len(s) < 30:
        return False

    skew = s.skew()
    kurt = s.kurt()
    return (abs(skew) < skew_thr) and (abs(kurt) < kurt_thr)

def rescale(df):
    sub = df.iloc[:, 2:]
    
    X = sub.to_numpy(dtype=float)
    X_scaled = np.zeros_like(X, dtype=float)

    for j in range(X.shape[1]):
        col = X[:, j]
        if normal(col):
            scaler = StandardScaler()
        else:
            scaler = MinMaxScaler()
        X_scaled[:, j] = scaler.fit_transform(col.reshape(-1, 1)).ravel()

    return X_scaled

def dbcv_score(
    X,
    labels,
    alpha_noise = 0.3,
    alpha_imbalance = 0.3,
    min_clusters = 2,
):
    n = len(labels)
    if n == 0:
        return -1e9

    noise_ratio = np.sum(labels == -1) / n

    cluster_labels = sorted(set(labels) - {-1})
    n_clusters = len(cluster_labels)
    if n_clusters < min_clusters:
        return -1e9

    sizes = np.array([(labels == c).sum() for c in cluster_labels], dtype=float)
    cv_sizes = sizes.std() / sizes.mean() if sizes.mean() > 0 else 0.0

    try:
        base = dbcv.dbcv(X, labels, metric="euclidean", noise_id=-1)
    except Exception:
        return -1e9

    score = base - alpha_noise * noise_ratio - alpha_imbalance * cv_sizes
    return float(score)

def optimize_dbscan(
    X: np.ndarray,
    eps_values,
    min_samples_values,
    metric = "euclidean",
    alpha_noise = 0.3,
    alpha_imbalance = 0.3,
):
    best_score = -np.inf
    best_params = None
    history = []

    for eps in eps_values:
        for min_samples in min_samples_values:
            model = DBSCAN(eps=eps, min_samples=min_samples, metric=metric)
            labels = model.fit_predict(X)

            score = dbcv_score(
                X,
                labels,
                alpha_noise=alpha_noise,
                alpha_imbalance=alpha_imbalance,
            )

            history.append(
                {
                    "eps": float(eps),
                    "min_samples": int(min_samples),
                    "score": float(score),
                }
            )

            if score > best_score:
                best_score = score
                best_params = {"eps": float(eps), "min_samples": int(min_samples)}

    return best_params, best_score, history

df = rescale(df)

eps_grid = np.linspace(0.2, 5.0, 30)
min_samples_grid = [3, 4, 5, 6, 8, 10, 15]

best_params, best_score, history = optimize_dbscan(
    df,
    eps_values=eps_grid,
    min_samples_values=min_samples_grid,
    metric="euclidean",
    alpha_noise=0.3,
    alpha_imbalance=0.3,
)

print("Mejores parámetros:", best_params)
print("Mejor score (DBCV penalizado):", best_score)

history_df = pd.DataFrame(history)
history_df.head()